In [4]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
spark.sql("""
    CREATE TABLE IF NOT EXISTS silver.customers (
    customer_id STRING,
    name STRING,
    email STRING,
    country STRING,
    customer_type STRING,
    registration_date DATE,
    age INT,
    gender STRING,
    total_purchases INT,
    customer_segment STRING,
    days_since_registration INT,
    last_updated TIMESTAMP)
""")

StatementMeta(, 911b0a2e-1afb-4d11-8912-33e48f2d7049, 6, Finished, Available, Finished, False)

DataFrame[]

In [5]:
last_processed_df = spark.sql("SELECT MAX(last_updated) as last_processed FROM silver.customers")

StatementMeta(, 911b0a2e-1afb-4d11-8912-33e48f2d7049, 7, Finished, Available, Finished, False)

In [6]:
last_processed_timestamp = last_processed_df.collect()[0]['last_processed']

StatementMeta(, 911b0a2e-1afb-4d11-8912-33e48f2d7049, 8, Finished, Available, Finished, False)

In [7]:
if last_processed_timestamp is None:
    last_processed_timestamp = "1900-01-01T00:00:00.000+00:00"

StatementMeta(, 911b0a2e-1afb-4d11-8912-33e48f2d7049, 9, Finished, Available, Finished, False)

In [12]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW incremental AS
SELECT *
FROM enterpriseRetail.retailLakehouse.bronze.customers
WHERE ingestion_timestamp > '{last_processed_timestamp}'
""")

StatementMeta(, 911b0a2e-1afb-4d11-8912-33e48f2d7049, 14, Finished, Available, Finished, False)

DataFrame[]

In [13]:
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW silver_incremental AS
SELECT
    customer_id,
    name,
    email,
    country,
    customer_type,
    registration_date,
    age,
    gender,
    total_purchases,
    CASE
        WHEN total_purchases > 10000 THEN 'High Value'
        WHEN total_purchases > 5000 THEN 'Medium Value'
        ELSE 'Low Value'
    END AS customer_segment,
    DATEDIFF(CURRENT_DATE(), registration_date) AS days_since_registration,
    CURRENT_TIMESTAMP() AS last_updated
FROM incremental
WHERE 
    age BETWEEN 18 AND 100
    AND email IS NOT NULL
    AND total_purchases >= 0
""")

StatementMeta(, 911b0a2e-1afb-4d11-8912-33e48f2d7049, 15, Finished, Available, Finished, False)

DataFrame[]

In [14]:
spark.sql("""
MERGE INTO silver.customers target
USING silver_incremental source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN
    UPDATE SET *
WHEN NOT MATCHED THEN
    INSERT *
""")

StatementMeta(, 911b0a2e-1afb-4d11-8912-33e48f2d7049, 16, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [15]:
# Read and verify the Silver layer customer data
spark.sql("select count(*) from silver.customers").show()

StatementMeta(, 911b0a2e-1afb-4d11-8912-33e48f2d7049, 17, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
|     930|
+--------+

